## Generation of Protein Vector Given a 2D Sequence

Import Packages:

In [1]:
import numpy as np

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

import re

from typing import Union

import transformers
from transformers import AutoTokenizer, AutoModel


import requests

import esm

from tscales_bert_cls import TScalesBERTEncoder, encode_tscales_cls

import pytorch_lightning as pl

from pytorch_lightning.loggers import WandbLogger



c:\Users\Adrian\Documents\Studium\M1\ai\AI-for-Chemistry-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load the dataset:

In [2]:
df = pd.read_csv("Datasets/Fluorescent-Protein-Database.csv")

Generate t-scale for protein sequences:

In [3]:
encoder = TScalesBERTEncoder(d_model=256, nhead=8, num_layers=4)
tscales_cls = encode_tscales_cls(df["Protein sequence"], encoder)
df["tscales_cls"] = list(tscales_cls)

c:\Users\Adrian\Documents\Studium\M1\ai\AI-for-Chemistry-Project\tscales_bert_cls.py:84: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer,


Using Evolutionary Scale Modeling (ESM-2) Embeddings

In [4]:
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()


In [5]:
def get_esm_embedding(seq):
    data = [("protein", seq)]

    _, _, tokens = batch_converter(data)

    with torch.no_grad():
        results = model(tokens, repr_layers=[33])

    # layer 33 = final representation
    reps = results["representations"][33]

    # remove special tokens and mean pool
    embedding = reps[0, 1:-1].mean(0)

    return embedding.numpy()


df["esm"] = df["Protein sequence"].apply(get_esm_embedding)

Encode SMILES of chromophores using pre-trained chemBERTa RNN

In [6]:
#assignment of canonical SMILES of ligand
smiles_dict = {
    "NRQ": r"CSCCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CRQ": r"NC(=O)CCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "NRP": r"CC(C)CC(=N)C1=NC(=C/c2ccc(O)cc2)/C(=O)N1CC(O)=O",
    "CH6": r"CSCC[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CRO": r"[C@@H](O)[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "5SQ": r"N[C@@H](Cc1c[nH]cn1)C2=NC(=C\c3ccc(O)cc3)/C(=O)N2CC(O)=O",
    "4M9": r"NC(=O)CCC(=N)C1=NC(=C\c2c[nH]c3ccccc23)/C(=O)N1CC(O)=O",
    "CR2": r"NCC1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "OFM": r"C[C@H]1O[C@@](O)(N=C1C2=N\C(=C/c3ccc(O)cc3)C(=O)N2CC(O)=O)[C@@H](N)Cc4ccccc4",
    "CR8": r"N[C@@H](Cc1[nH]cnc1)c2nc(C=C3C=CC(=O)C=C3)c([O-])n2CC(O)=O",
    "CFY": r"N[C@@H](Cc1ccccc1)[C@@]2(O)SCC(=N2)C3=NC(=C\c4ccc(O)cc4)/C(=O)N3CC(O)=O",
    "OIM": r"CC[C@H](C)[C@H](N)[C@@]1(O)O[C@H](C)C(=N1)C2=N\C(=C/c3ccc(O)cc3)C(=O)N2CC(O)=O",
    "CH7": r"OC(=O)CN1C(=O)C(=C/c2ccc(O)cc2)/N=C1C3=NCCCC3",
    "GYS": r"N[C@@H](CO)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "WCR": r"C[C@@]1(O)NC(=C\c2ccc(O)cc2)/C(=O)N1CC(O)=O",
    "DYG": r"N[C@@H](CC(O)=O)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "FAD": r"Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P@](O)(=O)O[P@@](O)(=O)OC[C@H]4O[C@H]([C@H](O)[C@@H]4O)n5cnc6c(N)ncnc56)c2cc1C",
    "PIA": r"C[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "BLR": r"CC1=C(C=C)C(/NC1=O)=C/c2[nH]c(Cc3[nH]c(\C=C4/NC(=O)C(=C4C)C=C)c(C)c3CCC(O)=O)c(CCC(O)=O)c2C",
    "CRF": r"C[C@@H](O)[C@H](N)C1=N\C(=C/c2c[nH]c3ccccc23)C(=O)N1CC(O)=O",
    "NYG": r"N[C@@H](CC(N)=O)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "FMN": r"Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P](O)(O)=O)c2cc1C",
    "B2H": r"C[C@@H](O)[C@H](N)c1nc(Cc2c[nH]c3ccccc23)c(O)n1CC(O)=O",
    "SWG": r"N[C@@H](CO)C1=N\C(=C/c2c[nH]c3ccccc23)C(=O)N1CC(O)=O",
    "CSH": r"N[C@@H](CO)[C@H]1N[C@@H](Cc2c[nH]cn2)C(=O)N1CC(O)=O",
    "BJF": r"CC(C)C[C@H](N)c1nc(CC(C)C)c(O)n1CC(O)=O",
    "GYC": r"N[C@@H](CS)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CCY": r"N[C@@H](CS)[C@H]1N[C@@H](Cc2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CR7": r"NCCCC[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O"
}   
df["smiles"] = df["Chromophore/ligand"].str.strip().str.upper().map(smiles_dict)
from transformers import AutoTokenizer, AutoModel


model_name = "seyonec/ChemBERTa-zinc-base-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

def smiles_to_vector(smiles):
    if not isinstance(smiles, str):
        return None

    inputs = tokenizer(smiles, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    # CLS token embedding = molecular representation
    pooled = outputs.last_hidden_state.max(dim=1).values.squeeze()

    return pooled.numpy()


df["smiles_vectors"] = df["smiles"].apply(smiles_to_vector)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 32597.40it/s]
[transformers] RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Concatenation of sequence vectors obtained with ESM, ligand SMILES vectors obtained with chemBERTa, and predictor values (Stokes shift, EC, quantum yield, protein mass) 

Define 2 model

In [7]:

def concat_model1(row):
    return np.concatenate([
        row["esm"],                    # 1280-d
        row["smiles_vectors"],         # 768-d
        np.array([
            row["Stokes shift"],      
            row["kDa"]
        ])
    ])



def concat_model2(row):
    return np.concatenate([
        row["tscales_cls"],            # 256-d
        row["smiles_vectors"],         # 768-d
        np.array([
            row["Stokes shift"],       
            row["kDa"]
        ])
    ])



Normalize and split

In [9]:

scaler = StandardScaler()
num_cols = ["Stokes shift", "kDa"]
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols]  = scaler.transform(test_df[num_cols])


Beyond concatenation, the future X and Y inputs to the NN must be brought into shape digestible to NN. N.b.: from this point on data in no longer stored in df, as it makes variation of shape more managable.

In [10]:
train_df["X_train_esm"] = train_df.apply(concat_model1, axis=1)
test_df["X_test_esm"]  = test_df.apply(concat_model1, axis=1)

train_df["X_train_tscale"] = train_df.apply(concat_model2, axis=1)
test_df["X_test_tscale"]  = test_df.apply(concat_model2, axis=1)

esm_train_stacked = np.stack(train_df["X_train_esm"].values).astype(np.float32)
esm_test_stacked = np.stack(test_df["X_test_esm"].values).astype(np.float32)
tscales_train_stacked = np.stack(train_df["X_train_tscale"].values).astype(np.float32)
tscales_test_stacked = np.stack(test_df["X_test_tscale"].values).astype(np.float32)
Emission_wavelength_train_float = train_df["Emission wavelength"].values.astype(np.float32)
Emission_wavelength_test_float = test_df["Emission wavelength"].values.astype(np.float32)

Input: X input will be concatenation of sequence vector, SMILES vector, stokes shift and molecular weight. Y input will be emission wavelenght.

In [11]:

X_train_esm = esm_train_stacked
X_test_esm = esm_test_stacked  

X_train_tscales = tscales_train_stacked 
X_test_tscales = tscales_test_stacked 

Y_train_esm = Emission_wavelength_train_float
Y_test_esm = Emission_wavelength_test_float

Y_train_tscales = Emission_wavelength_train_float
Y_test_tscales = Emission_wavelength_test_float

Using PyTorch, a Neural Network class is defined:

In [12]:
class NeuralNetwork(pl.LightningModule):
    def __init__(self, input_sz, hidden_sz, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()  # saves input_size, hidden_size, lr

        self.layer_1 = torch.nn.Linear(input_sz, hidden_sz)
        self.layer_2 = torch.nn.Linear(hidden_sz, hidden_sz)
        self.layer_3 = torch.nn.Linear(hidden_sz, 1)

    def forward(self, x):
        x = self.layer_1(x)
        x = F.relu(x)
        x = self.layer_2(x)
        x = F.relu(x)
        x = self.layer_3(x)
        return x # Shape: (batch_size, 1)
    
    def _shared_step(self, batch):
        x, y = batch
        preds = self(x)
        loss = F.mse_loss(preds, y)
        return loss
        
    def training_step(self, batch, batch_idx):
        loss = self._shared_step(batch)
        self.log("Train loss", loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        loss = self._shared_step(batch)
        self.log("Valid MSE", loss)

        
    def test_step(self, batch, batch_idx):
        loss = self._shared_step(batch)
        self.log("Test MSE", loss)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr
        )
        return optimizer

The Dataset class is defined as well:

In [13]:
from torch.utils.data import Dataset

class FPDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)
    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
esm_train_data = FPDataset(X_train_esm, Y_train_esm)
tscales_train_data = FPDataset(X_train_tscales, Y_train_tscales)
esm_test_data = FPDataset(X_train_esm, Y_train_esm)
tscales_test_data = FPDataset(X_train_tscales, Y_train_tscales)

Next, the datamodule is defined: 

In [38]:
class NeuralNetworkDataModule(pl.LightningDataModule):
    def __init__(self, train_dataset, val_dataset, test_dataset, batch_size=256):
        super().__init__()
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        self.batch_size = batch_size

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)
    
    def val_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)    

Finally, the model is trained. As the dataset is small (122 proteins), it was decided to proceed with k-fold cross-validation. This avoids a second split of the training dataset. First, the esm-encoded input is used:

In [51]:
batch_size = 32
hidden_size = 512
lr = 1e-3
max_epochs = 100
n_splits = 5

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
fold_mse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_esm)):

    X_tr, X_val = X_train_esm[train_idx], X_train_esm[val_idx]
    y_tr, y_val = Y_train_esm[train_idx], Y_train_esm[val_idx]

    train_data = FPDataset(X_tr, y_tr)
    val_data   = FPDataset(X_val, y_val)

    data_module = NeuralNetworkDataModule(
        train_dataset=esm_train_data,
        val_dataset=esm_train_data,
        test_dataset=esm_test_data,
        batch_size=batch_size
    )

    model = NeuralNetwork(
        input_sz=X_train_esm.shape[1],   
        hidden_sz=hidden_size,
        lr=lr
    )

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        logger=False,
        enable_checkpointing=True,
        enable_progress_bar=False
    )

    trainer.fit(model, datamodule=data_module)

    fold_mse.append(
        trainer.validate(model, datamodule=data_module, verbose=False)[0]["Valid MSE"]
    )

print("CV MSE mean:", np.mean(fold_mse))
print("CV MSE std:", np.std(fold_mse))

final_model_esm = NeuralNetwork(
        input_sz = X_train_esm.shape[1],
        hidden_sz=hidden_size,
        lr=lr
    )

final_data_module_esm = NeuralNetworkDataModule(
    train_dataset=esm_train_data,
    val_dataset=esm_train_data,
    test_dataset=esm_test_data,
    batch_size=batch_size
)
final_trainer_esm = pl.Trainer(
    max_epochs=max_epochs,
    logger=False,
    enable_checkpointing=True
)
final_trainer_esm.fit(final_model_esm, datamodule=final_data_module_esm)
results_esm = trainer.test(final_model_esm, datamodule=final_data_module_esm, ckpt_path="best")


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layer_1 │ Linear │  1.1 M │ train │     0 │
│ 1 │ layer_2 │ Linear │  262 K │ train │     0 │
│ 2 │ layer_3 │ Linear │    513 │ train │     0 │
└───┴─────────┴────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=100` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layer_1 │ Linear │  1.1 M │ train │     0 │
│ 1 │ layer_2 │ Linear │  262 K │ train │     0 │
│ 2 │ layer_3 │ Linear │    513 │ train │     0 │
└───┴─────────┴────────┴────────┴───────┴───────┘

...

┏━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layer_1 │ Linear │  1.1 M │ train │     0 │
│ 1 │ layer_2 │ Linear │  262 K │ train │     0 │
│ 2 │ layer_3 │ Linear │    513 │ train │     0 │
└───┴─────────┴────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=100` reached.


Restoring states from the checkpoint path at c:\Users\Adrian\Documents\Studium\M1\ai\AI-for-Chemistry-Project\checkpoints\epoch=99-step=400-v46.ckpt
Loaded model weights from the checkpoint at c:\Users\Adrian\Documents\Studium\M1\ai\AI-for-Chemistry-Project\checkpoints\epoch=99-step=400-v46.ckpt


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         Test MSE          │      977.38427734375      │
└───────────────────────────┴───────────────────────────┘

Now, the same is done using the tscale data:

In [52]:
batch_size = 32
hidden_size = 512
lr = 1e-3
max_epochs = 100
n_splits = 5

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
fold_mse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_esm)):

    X_tr, X_val = X_train_esm[train_idx], X_train_esm[val_idx]
    y_tr, y_val = Y_train_esm[train_idx], Y_train_esm[val_idx]

    train_data = FPDataset(X_tr, y_tr)
    val_data   = FPDataset(X_val, y_val)

    data_module = NeuralNetworkDataModule(
        train_dataset=tscales_train_data,
        val_dataset=tscales_train_data,
        test_dataset=tscales_test_data,
        batch_size=batch_size
    )

    model = NeuralNetwork(
        input_sz = 1026,
        hidden_sz=hidden_size,
        lr=lr
    )

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        logger=False,
        enable_checkpointing=True,
        enable_progress_bar=False
    )

    trainer.fit(model, datamodule=data_module)

    fold_mse.append(
        trainer.validate(model, datamodule=data_module, verbose=False)[0]["Valid MSE"]
    )

print("CV MSE mean:", np.mean(fold_mse))
print("CV MSE std:", np.std(fold_mse))

final_model_tscale = NeuralNetwork(
        input_sz = 1026,
        hidden_sz=hidden_size,
        lr=lr
    )

final_data_module = NeuralNetworkDataModule(
    train_dataset=tscales_train_data,
    val_dataset=tscales_train_data,
    test_dataset=tscales_test_data,
    batch_size=batch_size
)
final_trainer = pl.Trainer(
    max_epochs=max_epochs,
    logger=False,
    enable_checkpointing=True
)
final_trainer.fit(final_model_tscale, datamodule=final_data_module)
results_tscale = trainer.test(final_model_tscale, datamodule=final_data_module, ckpt_path="best")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layer_1 │ Linear │  525 K │ train │     0 │
│ 1 │ layer_2 │ Linear │  262 K │ train │     0 │
│ 2 │ layer_3 │ Linear │    513 │ train │     0 │
└───┴─────────┴────────┴────────┴───────┴───────┘

Trainable params: 788 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 788 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=100` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layer_1 │ Linear │  525 K │ train │     0 │
│ 1 │ layer_2 │ Linear │  262 K │ train │     0 │
│ 2 │ layer_3 │ Linear │    513 │ train │     0 │
└───┴─────────┴────────┴────────┴───────┴───────┘

...

┏━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layer_1 │ Linear │  525 K │ train │     0 │
│ 1 │ layer_2 │ Linear │  262 K │ train │     0 │
│ 2 │ layer_3 │ Linear │    513 │ train │     0 │
└───┴─────────┴────────┴────────┴───────┴───────┘

Trainable params: 788 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 788 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=100` reached.


Restoring states from the checkpoint path at c:\Users\Adrian\Documents\Studium\M1\ai\AI-for-Chemistry-Project\checkpoints\epoch=99-step=400-v52.ckpt
Loaded model weights from the checkpoint at c:\Users\Adrian\Documents\Studium\M1\ai\AI-for-Chemistry-Project\checkpoints\epoch=99-step=400-v52.ckpt


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         Test MSE          │    1315.9298095703125     │
└───────────────────────────┴───────────────────────────┘